# 05 — Patterns

Day-of-week and optional-location patterns for the selected period.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))

In [ ]:
# Inclusive reporting period, based on task start date.
START_DATE = "2024-01-01"
END_DATE = "9999-12-31"
REPORT_DATE = date.today()

query = construct_query("task-history.sql", {"START-DATE": START_DATE, "END-DATE": END_DATE})
history = normalise_history(query_data(query))
period_label = f"{START_DATE} to {END_DATE} ({len(history):,} tasks)"
print(f"Reporting period: {period_label}")


## Start day of week

In [ ]:
days = grouped_summary(history, "Start Day").sort_values("Start Day")
days

In [ ]:
days.plot.bar(x="Start Day", y="Task Count", legend=False, title=f"Tasks by start day — {period_label}")
plt.ylabel("Tasks"); plt.tight_layout(); plt.show()

## Category by day of week

In [ ]:
day_categories = history.pivot_table(index="Start Day", columns="Category", values="Task ID", aggfunc="count", fill_value=0, observed=False).reset_index()
day_categories

## Location

In [ ]:
locations = grouped_summary(history, "Location")
locations["Percentage With Recorded Location"] = percentage(int(history["Has Location"].eq(1).sum()), len(history))
locations

In [ ]:
EXPORT_NAME = "05-patterns.xlsx"
EXPORT_DATA = {"Day of Week": days, "Day and Category": day_categories, "Locations": locations}
export_to_spreadsheet(
    get_export_folder_path(), EXPORT_NAME, EXPORT_DATA
)
print(f"Exported {EXPORT_NAME} to {get_export_folder_path()}")
